In [ ]:

import numpy as np
from copy import deepcopy


In [ ]:

# Grid size (3 rows x 4 columns)
rows = 3
cols = 4

step_reward = -0.04
gamma = 0.9

# Terminal states (0-based indexing)
# +1 at top-right, -1 just below it
terminal_states = {
    (0, 3): 1,
    (1, 3): -1
}

# Blocked cell
blocked = (1, 1)

actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']

action_vectors = {
    'UP':    (-1, 0),
    'DOWN':  (1, 0),
    'LEFT':  (0, -1),
    'RIGHT': (0, 1)
}


In [ ]:

def left_of(action):
    return {
        'UP': 'LEFT',
        'DOWN': 'RIGHT',
        'LEFT': 'DOWN',
        'RIGHT': 'UP'
    }[action]

def right_of(action):
    return {
        'UP': 'RIGHT',
        'DOWN': 'LEFT',
        'LEFT': 'UP',
        'RIGHT': 'DOWN'
    }[action]

def get_next_state(state, action):
    r, c = state
    dr, dc = action_vectors[action]
    nr, nc = r + dr, c + dc
    
    if nr < 0 or nr >= rows or nc < 0 or nc >= cols or (nr, nc) == blocked:
        return state
    return (nr, nc)

def get_transitions(state, action):
    return [
        (0.8, get_next_state(state, action)),
        (0.1, get_next_state(state, left_of(action))),
        (0.1, get_next_state(state, right_of(action)))
    ]


In [ ]:

V = np.zeros((rows, cols))

for (r, c), reward in terminal_states.items():
    V[r, c] = reward

def is_terminal(state):
    return state in terminal_states


In [ ]:

def print_values(V):
    print("State Values:")
    for r in range(rows):
        for c in range(cols):
            if (r, c) == blocked:
                print("  XXXX  ", end=" ")
            else:
                print(f"{V[r,c]:7.3f}", end=" ")
        print()
    print()

def print_policy(policy):
    print("Policy:")
    for r in range(rows):
        for c in range(cols):
            if (r, c) == blocked:
                print("  XXXX  ", end=" ")
            elif (r, c) in terminal_states:
                print(" TERMIN ", end=" ")
            else:
                print(f"{policy[(r,c)]:7}", end=" ")
        print()
    print()


In [ ]:

theta = 1e-4
iteration = 0

while True:
    delta = 0
    new_V = deepcopy(V)
    policy = {}
    
    print(f"\n========== ITERATION {iteration} ==========\n")
    
    for r in range(rows):
        for c in range(cols):
            state = (r, c)
            
            if state == blocked or is_terminal(state):
                continue
            
            action_values = {}
            
            for action in actions:
                total = 0
                for prob, next_state in get_transitions(state, action):
                    reward = terminal_states.get(next_state, step_reward)
                    total += prob * (reward + gamma * V[next_state])
                
                action_values[action] = total
            
            best_action = max(action_values, key=action_values.get)
            best_value = action_values[best_action]
            
            new_V[r, c] = best_value
            policy[state] = best_action
            
            print(f"State {state}")
            for a in action_values:
                print(f"  Action {a:5} -> {action_values[a]:.4f}")
            print(f"  Best: {best_action} = {best_value:.4f}\n")
            
            delta = max(delta, abs(V[r,c] - best_value))
    
    V = new_V
    print_values(V)
    print_policy(policy)
    
    iteration += 1
    
    if delta < theta:
        print("Converged!")
        break
